In [11]:
# Plan: build vectorless RAG with the PageIndex cloud first, then build it locally ourselves
print("First we are creating the vectorless RAG using the cloud, after that we will create it locally")

First we are creating the vectorless RAG using the cloud, after that we will create it locally


In [12]:
# Load API keys from the .env file into environment variables
import os
from dotenv import load_dotenv

load_dotenv()  # searches for .env starting here and moving up the folders

# Check that both keys are loaded (prints only loaded/MISSING, never the key itself)
for name in ["OPENROUTER_API_KEY", "PAGEINDEX_API_KEY"]:
    value = os.getenv(name)
    print(name, "->", "loaded" if value else "MISSING")

OPENROUTER_API_KEY -> loaded
PAGEINDEX_API_KEY -> loaded


In [13]:
# Create the PageIndex client
# index="cloud" -> PageIndex servers build the tree (uses PAGEINDEX_API_KEY from env)
# chat=...      -> this OpenRouter model answers questions (uses OPENROUTER_API_KEY from env)
from pageindex import PageIndexClient

client = PageIndexClient(
    index="cloud",
    chat="openrouter/nvidia/nemotron-3-ultra-550b-a55b:free",
)

In [14]:
# Get the document id
# The PDF is already uploaded, so reuse its id (re-uploading would index it again and use credits)
# Note: it was uploaded twice, so your PageIndex dashboard shows "Attention.pdf" and "Attention_1.pdf". This id is the newer one.
doc_id = "pi-cmwaj1rxf002k0cnvz5fmrjsl"

# Run this only for a NEW PDF: uploads it and waits until the tree is built
# doc_id = client.submit_document("../../../data/Attention.pdf", wait=True)["doc_id"]

print(doc_id)

pi-cmwaj1rxf002k0cnvz5fmrjsl


In [15]:
# Fetch the tree from PageIndex
# node_summary=True  -> include each section's summary
# include_text=False -> skip full page text, keeps it small and readable
tree = client.get_tree(doc_id, node_summary=True, include_text=False)["result"]

# Print the tree as an indented outline (calls itself for each node's children)
def show(nodes, depth=0):
    for n in nodes:
        print("    " * depth + f"[{n['node_id']}] {n['title']}  (pages {n['start_index']}-{n['end_index']})")
        show(n.get("nodes", []), depth + 1)

show(tree)

[0000] Attention Is All You Need  (pages 1-15)
    [0001] Attention Is All You Need (intro)  (pages 1-1)
    [0002] Abstract  (pages 1-2)
    [0003] 1 Introduction  (pages 2-2)
    [0004] 2 Background  (pages 2-2)
    [0005] 3 Model Architecture  (pages 2-6)
        [0006] 3 Model Architecture (intro)  (pages 2-3)
        [0007] 3.1 Encoder and Decoder Stacks  (pages 3-3)
        [0008] 3.2 Attention  (pages 3-5)
            [0009] 3.2 Attention (intro)  (pages 3-4)
            [0010] 3.2.1 Scaled Dot-Product Attention  (pages 4-4)
            [0011] 3.2.2 Multi-Head Attention  (pages 4-5)
            [0012] 3.2.3 Applications of Attention in our Model  (pages 5-5)
        [0013] 3.3 Position-wise Feed-Forward Networks  (pages 5-5)
        [0014] 3.4 Embeddings and Softmax  (pages 5-6)
        [0015] 3.5 Positional Encoding  (pages 6-6)
    [0016] 4 Why Self-Attention  (pages 6-7)
    [0017] 5 Training  (pages 7-8)
    [0018] 6 Results  (pages 8-10)
        [0019] 6.1 Machine Translati

In [16]:
# Save the tree to results/ as a JSON file so we can open it in VS Code
import json

with open("../../../results/attention_tree.json", "w", encoding="utf-8") as f:
    # indent=2           -> 2 spaces per nesting level, so the file is readable
    # ensure_ascii=False -> keep characters like "Ł" as-is instead of \u0141
    json.dump(tree, f, indent=2, ensure_ascii=False)

In [17]:
# Ask a question: the model reads the tree, picks sections, reads those pages, then answers
answer = client.chat("What is this document about?", doc_id=doc_id)
print(answer)

This is the seminal **"Attention Is All You Need"** paper (Vaswani et al., 2017) that introduced the **Transformer architecture** — the foundation of modern LLMs like GPT, BERT, and their descendants.

## Core Contribution
The paper proposes a novel sequence transduction model that **entirely replaces recurrent (RNN/LSTM/GRU) and convolutional layers with self-attention mechanisms**. This eliminates the sequential computation bottleneck of RNNs, enabling massive parallelization during training.

## Key Architecture Components

| Component | Details |
|-----------|---------|
| **Overall Structure** | Encoder-decoder with 6 identical layers each |
| **Self-Attention** | Scaled Dot-Product Attention + Multi-Head Attention (8 heads) |
| **Feed-Forward** | Position-wise FFN (512 → 2048 → 512 with ReLU) |
| **Positional Encoding** | Sinusoidal functions (fixed, not learned) to inject sequence order |
| **Residual + LayerNorm** | Around every sub-layer |
| **Decoder Masking** | Causal masking

In [18]:
# Same question with streaming: prints the answer as it is written,
# plus the steps ([thinking], [tool_call]) showing which pages the model chose to read
for chunk in client.chat("What is this document about?", doc_id=doc_id, stream=True):
    print(chunk, end="", flush=True)

[thinking] TheThe user is user is asking what asking what the document " the document "Attention_1.pdf"Attention_1.pdf" is about. I is about. I have the metadata have the metadata which which says it introduces says it introduces the Transformer architecture. the Transformer architecture. But they But they want me to retrieve its content want me to retrieve its content using get_document_structure and using get_document_structure and get_page_content to give get_page_content to give a more detailed answer. Let a more detailed answer. Let me start me start by getting the document structure. by getting the document structure.

[tool_call] get_document_structure {"doc_name": "Attention_1.pdf"}
[tool_result] get_document_structure: { "success": true, "doc_name": "Attention_1.pdf", "structure": [ { "title": "Attention Is All You Need", "node_id": "0000", "start_index": 1, "end_index": 15, "summary": "This section, titled \"Attent... (+11704 chars)

[thinking] TheThe document structure shows

In [10]:
# Retrieval test: a specific question
# Check the [tool_call] lines: the right section is 3.2.2 Multi-Head Attention (pages 4-5)
# Result: the cloud model opened exactly pages 4-5, so the cloud tree led it straight to the right section
for chunk in client.chat("How does multi-head attention work?", doc_id=doc_id, stream=True):
    print(chunk, end="", flush=True)

[thinking] TheThe user is user is asking about how multi-head attention asking about how multi-head attention works, and works, and they've specified they've specified the document "Attention_ the document "Attention_1.pdf" which appears to1.pdf" which appears to be about the Transformer architecture be about the Transformer architecture. I should first get. I should first get the document structure to understand the the document structure to understand the layout, then extract layout, then extract relevant pages about relevant pages about multi-head attention. multi-head attention.

[tool_call] get_document_structure {"doc_name": "Attention_1.pdf"}
[tool_result] get_document_structure: { "success": true, "doc_name": "Attention_1.pdf", "structure": [ { "title": "Attention Is All You Need", "node_id": "0000", "start_index": 1, "end_index": 15, "summary": "This section, titled \"Attent... (+11704 chars)

[thinking] TheThe document structure shows that section document structure shows tha

# Part 2: PageIndex local (no PageIndex key)

Same SDK, but the tree is built **on our machine** by our own LLM through OpenRouter.
The only change is `index=`: a **model name** instead of `"cloud"`.

The tree is saved to disk: `results/pageindex_local/docs/<doc_id>/tree.json`
Comparison with the cloud version: [notes/pageindex_cloud_vs_local.md](../../../notes/pageindex_cloud_vs_local.md)

In [2]:
# LOCAL client: no PageIndex key needed (PageIndexClient is already imported in the cloud part)
# index = a dict with a model name (not "cloud") -> this model builds the tree on our machine
#   "storage_path" -> folder where tree.json, pages.json and doc.json are saved
# chat = the model that reads the tree and answers questions
local_client = PageIndexClient(
    index={
        "model": "openrouter/nvidia/nemotron-3-ultra-550b-a55b:free",
        "storage_path": "../../../results/pageindex_local",
    },
    chat="openrouter/nvidia/nemotron-3-ultra-550b-a55b:free",
)

In [6]:
# The PDF is already indexed locally, so reuse its id
# (running submit_document again would rebuild the tree and spend OpenRouter calls)
local_doc_id = "pi-674f81e1daac4d75a9e3c7aed9d2949f"

# Run this only for a NEW PDF: builds the tree locally and waits until it is done
# local_doc_id = local_client.submit_document("../../../data/Attention.pdf", wait=True)["doc_id"]

print(local_doc_id)

pi-674f81e1daac4d75a9e3c7aed9d2949f


In [8]:
# Read the local tree from results/pageindex_local/ (same get_tree call as the cloud part)
# node_summary=True -> include summaries, include_text=False -> skip full page text
local_tree = local_client.get_tree(local_doc_id, node_summary=True, include_text=False)["result"]

# show() is defined in the cloud tree cell above, so we just reuse it
show(local_tree)

# Compared with the cloud tree: flatter (14 nodes vs 25), section numbers lost,
# "1 Introduction" and "2 Background" wrongly nested under "Model Architecture",
# and "Conclusion" (pages 10-15) also swallows References + Attention Visualizations

[0000] Abstract  (pages 1-2)
[0001] Introducing the Transformer Model Architecture  (pages 2-2)
[0002] Model Architecture  (pages 2-6)
    [0003] 1 Introduction  (pages 2-2)
    [0004] 2 Background  (pages 2-3)
    [0005] Encoder and Decoder Stacks  (pages 3-3)
    [0006] Attention  (pages 3-5)
    [0007] Position-wise Feed-Forward Networks  (pages 5-5)
    [0008] Embeddings and Softmax  (pages 5-6)
    [0009] Positional Encoding  (pages 6-6)
[0010] Why Self-Attention  (pages 6-7)
[0011] Training  (pages 7-8)
[0012] Results  (pages 8-10)
[0013] Conclusion  (pages 10-15)


In [9]:
# Same retrieval test as the cloud part, so we can compare
# Result: the local model opened pages 3-5 (the whole "Attention" node), not just 4-5,
# because the local tree has no separate "Multi-Head Attention" node -> it reads more pages
for chunk in local_client.chat("How does multi-head attention work?", doc_id=local_doc_id, stream=True):
    print(chunk, end="", flush=True)

[thinking] TheThe user is user is asking about how multi-head attention asking about how multi-head attention works in works in the context of the " the context of the "Attention.pdf" document,Attention.pdf" document, which is the Transformer paper which is the Transformer paper. I need to retrieve. I need to retrieve the document structure first to find the document structure first to find the relevant sections, then get the relevant sections, then get the page content for those the page content for those sections. sections.

[tool_call] get_document_structure {"doc_name": "Attention.pdf"}
[tool_result] get_document_structure: {"success": true, "doc_name": "Attention.pdf", "structure": [{"title": "Abstract", "node_id": "0000", "start_index": 1, "end_index": 2, "summary": "The paper introduces the Transformer, a novel neural... (+17551 chars)

[thinking] TheThe document structure shows that the " document structure shows that the "Attention" section (nodeAttention" section (node_id: "_

## Cloud vs Local: what we saw

| | Cloud | Local |
|---|---|---|
| Tree nodes / depth | 25 nodes, 4 levels | 14 nodes, 2 levels |
| Pages opened for *"How does multi-head attention work?"* | 4-5 (exact section 3.2.2) | 3-5 (whole "Attention" section) |
| Keys used | PageIndex + OpenRouter | OpenRouter only |

**Lesson:** the model can only pick sections that exist in the tree, so **a better tree gives more precise retrieval**.
Full details: [notes/pageindex_cloud_vs_local.md](../../../notes/pageindex_cloud_vs_local.md)